In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyApp") \
    .getOrCreate()

# Spark UDF (User Defined Function)

In [3]:
from pyspark.sql.functions import col

In [4]:
data = [
    ("U1", 85),
    ("U2", 72),
    ("U3", 40)
]

In [5]:
df=spark.createDataFrame(data, ["user_id", "score"])

In [6]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

In [7]:
def classify(score):
  if score>80:
    return "High"
  elif score >=50:
    return "Medium"
  else:
    return "Low"

In [15]:
p_udf=udf(classify, StringType())

In [16]:
df.withColumn("performace", p_udf("score")).show()

+-------+-----+----------+
|user_id|score|performace|
+-------+-----+----------+
|     U1|   85|      High|
|     U2|   72|    Medium|
|     U3|   40|       Low|
+-------+-----+----------+



In [17]:
df=df.withColumn("performace", p_udf("score"))

In [18]:
df.withColumn("performace", p_udf("score")).explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(performace, classify('score)#46, None)]
+- Project [user_id#0, score#1L, classify(score#1L)#44 AS performace#45]
   +- Project [user_id#0, score#1L, classify(score#1L)#29 AS performace#30]
      +- LogicalRDD [user_id#0, score#1L], false

== Analyzed Logical Plan ==
user_id: string, score: bigint, performace: string
Project [user_id#0, score#1L, classify(score#1L)#46 AS performace#47]
+- Project [user_id#0, score#1L, classify(score#1L)#44 AS performace#45]
   +- Project [user_id#0, score#1L, classify(score#1L)#29 AS performace#30]
      +- LogicalRDD [user_id#0, score#1L], false

== Optimized Logical Plan ==
Project [user_id#0, score#1L, pythonUDF0#48 AS performace#47]
+- BatchEvalPython [classify(score#1L)#46], [pythonUDF0#48]
   +- LogicalRDD [user_id#0, score#1L], false

== Physical Plan ==
*(2) Project [user_id#0, score#1L, pythonUDF0#48 AS performace#47]
+- BatchEvalPython [classify(score#1L)#46], [pythonUDF0#48]
   +- 

# PySpark Native Way

In [10]:
from pyspark.sql.functions import when
df.withColumn(
    "performance",
    when(col("score")>=80, "High")
    .when(col("score")>=50, "Medium")
    .otherwise("Low")
).show()

+-------+-----+-----------+
|user_id|score|performance|
+-------+-----+-----------+
|     U1|   85|       High|
|     U2|   72|     Medium|
|     U3|   40|        Low|
+-------+-----+-----------+



#Set Operations (Row-level)

In [24]:
data_a=[
    ("UI", "Python"),
    ("U2", "java"),
    ("U3", "Spark"),
]

df_a=spark.createDataFrame(data_a, ["user_id", "course"])

In [25]:
data_b=[
    ("U7", "PySpark"),
    ("U2", "java"),
    ("U3", "Spark"),
]

df_b=spark.createDataFrame(data_b, ["user_id", "course"])

In [26]:
df_a.union(df_b).show()

+-------+-------+
|user_id| course|
+-------+-------+
|     UI| Python|
|     U2|   java|
|     U3|  Spark|
|     U7|PySpark|
|     U2|   java|
|     U3|  Spark|
+-------+-------+



In [27]:
df_a.intersect(df_b).show()

+-------+------+
|user_id|course|
+-------+------+
|     U2|  java|
|     U3| Spark|
+-------+------+

